# Full testbed

In [7]:
import dataclasses
import logging
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from anb.agents import ANBAgent
from anb.metrics import RESPONSE_COLUMNS
from anb.model import ModelSpec
from anb.networks import ErdosRenyiNetwork, Network
from anb.testbed import PressureSchedule, Testbed, TestbedConfig

## Set config

In [8]:
N_AGENTS = 100
M = 10
TAU = 1
LINK_PROB = 10 / N_AGENTS
BETA = 3.0
RHO = 1/3

# The sweep from ANBmodel.py's __main__ block, one column per setting:
#   static low w | static high w | adaptive until external pressure | fully adaptive
SETTINGS = {
    "init_w":       [0.2,   0.8,   0.2,  0.2],
    "eps":          [0.0,   0.0,   1.0,  1.0],
    "mu":           [0.0,   0.0,   0.0,  0.0],
    "fixedBNat100": [False, False, True, False],
}
PRESSURES = [0, 1, 2, 4, 8, 16]
SEEDS = range(100)

# Legacy-named CSVs in ../simOut, where the analysis scripts look (ANBmodel.py writes to ../simOut_old_model)
OUT_DIR = Path("../simOut")
# True resumes a crashed sweep by skipping seeds whose CSV exists. Those CSVs are stale after a model change,
# so clear or switch OUT_DIR then.
SKIP_EXISTING = False


In [ ]:
# Timestamped INFO log, to the notebook output and to sweep.log next to the CSVs
OUT_DIR.mkdir(parents=True, exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout), logging.FileHandler(OUT_DIR / "sweep.log")],
    force=True,
)
log = logging.getLogger("testbed_notebook")


def make_spec(init_w, eps, mu, fixedBNat100):
    # fixedBNat100 is learning_until = 101 in the package (see docu/validation.md, deviation 2)
    agent = ANBAgent(beta=BETA, eps=eps, mu=mu, init_w=init_w, tau=TAU, n_beliefs=M,
                     learning_until=101 if fixedBNat100 else None)
    return ModelSpec(agent=agent, network_cls=ErdosRenyiNetwork,
                     network_kwargs={"link_prob": LINK_PROB}, n_agents=N_AGENTS, rho=RHO)


# One ensemble per (setting, pressure), like legacy's param_combis × seeds. Each seed's CSV is written as soon as it
# finishes, so a crash keeps everything done so far. n_jobs=-2 = all cores but one.
settings = [dict(zip(SETTINGS, values)) for values in zip(*SETTINGS.values())]
ensembles = [(setting, pressure) for setting in settings for pressure in PRESSURES]
for i, (setting, pressure) in enumerate(ensembles, start=1):
    log.info("ensemble %d/%d: %s, pressure=%s", i, len(ensembles), setting, pressure)
    config = TestbedConfig(PressureSchedule.single_event(strength=pressure))
    Testbed(make_spec(**setting), config).save_ensemble(SEEDS, OUT_DIR, n_jobs=-2, skip_existing=SKIP_EXISTING)